# 04 — Power Budget

Multi-rail power architecture for avionics + payload. Propulsion is
tracked separately via notebook 03.


In [ ]:
"""Add the repo root to sys.path so `import models` works from any subdir."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

FIG_DIR = REPO_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from models.power_model import PowerBudget

pb = PowerBudget()
pb.add_rail("main_6s",   voltage_v=22.2, capacity_a=None)  # propulsion pack
pb.add_rail("reg_5v",    voltage_v=5.0,  capacity_a=8.0)
pb.add_rail("reg_12v",   voltage_v=12.0, capacity_a=5.0)
pb.add_rail("sensor_5v", voltage_v=5.0,  capacity_a=4.0)


## Avionics + payload consumers


In [ ]:
# Edit datasheet values here as hardware is selected.
pb.add_consumer("pixhawk_6x",        "reg_5v",    idle_a=0.20, peak_a=0.40)
pb.add_consumer("rc_rx_elrs",        "reg_5v",    idle_a=0.05, peak_a=0.10)
pb.add_consumer("rtk_h_rtk_f9p",     "reg_5v",    idle_a=0.15, peak_a=0.20)
pb.add_consumer("orin_nano_carrier", "reg_12v",   idle_a=0.50, peak_a=1.50, duty_cycle=0.7)
pb.add_consumer("livox_mid360",      "reg_12v",   idle_a=0.30, peak_a=0.75, duty_cycle=1.0)
pb.add_consumer("flir_boson_plus",   "sensor_5v", idle_a=0.30, peak_a=0.50)
pb.add_consumer("global_shutter_rgb","sensor_5v", idle_a=0.40, peak_a=0.80)

import pandas as pd
totals = pd.DataFrame(pb.rail_totals()).T
totals.round(3)


## BEC / UBEC sizing recommendation


In [ ]:
recs = pb.bec_recommendation(headroom=0.20)
for rail, a in recs.items():
    print(f"{rail:<10s} -> {a:.2f} A continuous (peak * 1.20)")

warnings = pb.warnings()
if warnings:
    print("\nWARNINGS:")
    for w in warnings:
        print(" -", w)
else:
    print("\nNo rail capacity violations.")


## Hand-off: total avionics+payload power


In [ ]:
power_summary = {
    "avionics_payload_average_w": pb.total_average_w(),
    "rail_totals": pb.rail_totals(),
}
power_summary
